# Re-reading / System 2 Attention

Read twice, isolate relevant facts, and solve the reduced problem.

This executed notebook uses only public prompts, responses, and aggregate analysis. It does not embed answer keys or row-level correctness labels.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

root = Path.cwd().resolve()
manifest_path = root / 'tier1_manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8')) if manifest_path.exists() else {}

def read_public_csv(path):
    if not path.exists():
        return pd.DataFrame()
    frame = pd.read_csv(path, keep_default_na=False)
    forbidden = {'gold', 'ground_truth', 'correct_answer', 'normalized_gold', 'is_correct'}
    exposed = forbidden.intersection(str(column).lower() for column in frame.columns)
    if exposed:
        raise ValueError(f'public analysis contains forbidden row-level columns: {sorted(exposed)}')
    return frame

print('model:', manifest.get('model', 'not prepared'))
print('reasoning effort:', manifest.get('reasoning_effort', 'not prepared'))
print('prompt version:', manifest.get('prompt_version', 'not prepared'))


model: gpt-5.6-sol
reasoning effort: high
prompt version: gpt56-tier1-v1


In [2]:
condition = 'rereading_s2a'
queues = ('rereading_s2a',)
coverage = []
result_frames = []
for queue in queues:
    dataset_path = root / 'Dataset' / f'{queue}.csv'
    result_path = root / 'Result' / f'{queue}.csv'
    dataset = read_public_csv(dataset_path)
    result = read_public_csv(result_path)
    answered = 0
    if not result.empty and 'model_response' in result:
        answered = int(result['model_response'].astype(str).str.strip().ne('').sum())
        result_frames.append(result.assign(source_queue=queue))
    coverage.append({
        'queue': queue,
        'dataset_rows': len(dataset),
        'result_rows': len(result),
        'answered_rows': answered,
        'unresolved_rows': max(len(result) - answered, 0),
    })

display(pd.DataFrame(coverage))
if result_frames:
    results = pd.concat(result_frames, ignore_index=True)
    group_columns = [column for column in ('task', 'variant', 'stage', 'path_index') if column in results]
    display(
        results.groupby(group_columns, dropna=False)
        .agg(responses=('response_id', 'count'), answered=('model_response', lambda values: values.astype(str).str.strip().ne('').sum()))
        .reset_index()
    )
else:
    print('Result files are not available yet.')

analysis_dir = root / 'Analysis'
shown = 0
if analysis_dir.exists():
    for path in sorted(analysis_dir.glob('*.csv')):
        frame = read_public_csv(path)
        if frame.empty:
            continue
        selector = None
        if 'condition' in frame.columns:
            selector = frame['condition'].astype(str).eq(condition)
        elif 'method' in frame.columns:
            selector = frame['method'].astype(str).eq(condition)
        if selector is not None and bool(selector.any()):
            print(path.name)
            selected = frame.loc[selector].reset_index(drop=True)
            if 'evaluation_id' in selected.columns and len(selected) > 100:
                print('public row-level aggregate rows:', len(selected))
                grouping = [column for column in ('task', 'variant', 'agreement', 'status', 'answer_changed') if column in selected]
                if grouping:
                    display(selected.groupby(grouping, dropna=False).size().rename('rows').reset_index())
            else:
                display(selected)
            shown += 1
if not shown:
    print('No condition-level public analysis tables are available yet.')


,queue,dataset_rows,result_rows,answered_rows,unresolved_rows
0,rereading_s2a,200,200,200,0


,task,variant,stage,path_index,responses,answered
0,obfus_blood_relation,base,single,,50,50
1,obfus_blood_relation,obfuscation_l2,single,,50,50
2,obfus_fol,base,single,,50,50
3,obfus_fol,obfuscation,single,,50,50


accuracy_by_cell.csv


,model,condition,task,task_title,variant,correct,total,accuracy_pct,response_or_consensus_failures
0,gpt-5.6-sol,rereading_s2a,obfus_fol,FOL,base,42,50,84.0,0
1,gpt-5.6-sol,rereading_s2a,obfus_fol,FOL,obfuscation,35,50,70.0,0
2,gpt-5.6-sol,rereading_s2a,obfus_blood_relation,Blood Relation,base,38,50,76.0,0
3,gpt-5.6-sol,rereading_s2a,obfus_blood_relation,Blood Relation,obfuscation_l2,36,50,72.0,0


base_obfuscated_gaps.csv


...

,condition,scope,correct,total,accuracy_pct
0,rereading_s2a,all applicable variants,151,200,75.5
1,rereading_s2a,obfuscated variants,71,100,71.0


improvement_over_control.csv


,condition,task,variant,method_correct,control_correct,total,method_accuracy_pct,control_accuracy_pct,improvement_pp
0,rereading_s2a,obfus_fol,obfuscation,35,30,50,70.0,60.0,10.0
1,rereading_s2a,obfus_blood_relation,obfuscation_l2,36,37,50,72.0,74.0,-2.0


paired_retention.csv


,condition,task,variant,both_correct,base_only_correct,obfuscated_only_correct,both_wrong,total,paired_retention_pct
0,rereading_s2a,obfus_fol,obfuscation,32,10,3,5,50,76.1905
1,rereading_s2a,obfus_blood_relation,obfuscation_l2,36,2,0,12,50,94.7368
